In [15]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 1 │ Imports & Hardware Setup
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import gc
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
 
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [16]:
# ── Multi-GPU Setup ──────────────────────────────────────────────────────────
# DataParallel splits each mini-batch across available GPUs automatically.
# PRIMARY is where tensors live; DataParallel handles the rest.
N_GPUS  = min(torch.cuda.device_count(), 2)
PRIMARY = torch.device("cuda:0")
scaler_amp = GradScaler(device='cuda')
 
print(f"🚀 Using {N_GPUS} GPU(s):")
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"   [{i}] {p.name}  ({p.total_memory / 1e9:.1f} GB)")
 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

🚀 Using 2 GPU(s):
   [0] NVIDIA GeForce RTX 2080 Ti  (11.3 GB)
   [1] NVIDIA GeForce RTX 2080 Ti  (11.3 GB)


In [17]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 2 │ Data Loading & Storm-Aware Train / Val / Test Split
# ─────────────────────────────────────────────────────────────────────────────
# KEY HYDROLOGICAL RULE: splits happen ONLY at storm boundaries.
# A partial storm must belong entirely to one partition. This prevents
# the model from seeing the rising limb of a storm during training and
# predicting its recession limb during evaluation — a form of data leakage
# that inflates scores and doesn't reflect operational forecasting.
 
FILE_PATH = "Data_Files/delineated_storms.parquet"
 
FEATURES = [
    'precip_1hr [inch]',             # recent accumulation — direct flood driver
    'precip_max_intensity [inch/hour]', # peak intensity — controls runoff generation
    'temp_2m [degF]',                # governs snowmelt contribution
    'soil_moisture_05cm [m^3/m^3]', # antecedent moisture — critical for infiltration
    'elevation [feet]',              # controls flow routing and ponding
]
TARGET    = 'depth_inches'
TV_SPLIT  = (0.70, 0.15, 0.15)      # train / val / test fractions by storm count
 
df = pd.read_parquet(FILE_PATH)

In [18]:
# ── Resolve storm identifier column ─────────────────────────────────────────
STORM_COL = None
for candidate in ['storm_id', 'event_id', 'storm', 'event']:
    if candidate in df.columns:
        STORM_COL = candidate
        break
 
if STORM_COL is None:
    # Fall back: infer storms from gaps in the datetime index.
    # A gap > 6 h with no precipitation is a standard hydrological inter-event
    # criterion (e.g., USEPA, 1986). Adjust threshold for your sensor cadence.
    if isinstance(df.index, pd.DatetimeIndex):
        gap_seconds = df.index.to_series().diff().dt.total_seconds().fillna(0)
        df['_storm_id'] = (gap_seconds > 6 * 3600).cumsum()
    else:
        # Last resort: chunk sequentially (not ideal, but safe)
        CHUNK = 500
        df['_storm_id'] = np.arange(len(df)) // CHUNK
    STORM_COL = '_storm_id'
    print(f"⚠️  No storm ID column found. Inferred {df[STORM_COL].nunique()} events "
          f"from time-gap criterion (>6 h).")
else:
    print(f"✅ Storm column '{STORM_COL}' — {df[STORM_COL].nunique()} events detected.")
 

✅ Storm column 'storm_id' — 418 events detected.


In [19]:
# ── Drop incomplete rows, cast to float32 ───────────────────────────────────
df_clean = df[FEATURES + [TARGET, STORM_COL]].dropna().copy()
df_clean[FEATURES + [TARGET]] = df_clean[FEATURES + [TARGET]].astype('float32')
 
# ── Chronological storm-level split ─────────────────────────────────────────
# pandas unique() preserves insertion order → chronological if data is sorted.
storm_ids = df_clean[STORM_COL].unique()
n_storms  = len(storm_ids)
n_tr      = int(n_storms * TV_SPLIT[0])
n_va      = int(n_storms * TV_SPLIT[1])
 
train_storms = storm_ids[:n_tr]
val_storms   = storm_ids[n_tr : n_tr + n_va]
test_storms  = storm_ids[n_tr + n_va :]
 
train_df = df_clean[df_clean[STORM_COL].isin(train_storms)].copy()
val_df   = df_clean[df_clean[STORM_COL].isin(val_storms)].copy()
test_df  = df_clean[df_clean[STORM_COL].isin(test_storms)].copy()
 
print(f"\n📊 Storm-aware partition (no mid-storm cuts):")
print(f"   Train : {len(train_df):>8,} rows  ({len(train_storms):>4} storms)")
print(f"   Val   : {len(val_df):>8,} rows  ({len(val_storms):>4} storms)")
print(f"   Test  : {len(test_df):>8,} rows  ({len(test_storms):>4} storms)")


📊 Storm-aware partition (no mid-storm cuts):
   Train : 5,128,938 rows  ( 210 storms)
   Val   :  407,823 rows  (  45 storms)
   Test  :  120,584 rows  (  46 storms)


In [20]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 3 │ Scaling & GPU Tensor Push
# ─────────────────────────────────────────────────────────────────────────────
# Scalers fitted ONLY on train to avoid val/test statistics leaking into
# normalisation — a subtle but real form of data snooping.
 
scaler_X = StandardScaler()
scaler_y = StandardScaler()
 
X_tr  = scaler_X.fit_transform(train_df[FEATURES]).astype('float32')
X_val = scaler_X.transform(val_df[FEATURES]).astype('float32')
X_te  = scaler_X.transform(test_df[FEATURES]).astype('float32')
 
y_tr  = scaler_y.fit_transform(train_df[[TARGET]]).astype('float32')
y_val = scaler_y.transform(val_df[[TARGET]]).astype('float32')
 
# Raw (un-scaled) targets for metric evaluation
y_val_raw = val_df[TARGET].values.astype('float32')
y_te_raw  = test_df[TARGET].values.astype('float32')
 
# Storm ID arrays (CPU numpy — used by window builder)
sid_tr  = train_df[STORM_COL].values
sid_val = val_df[STORM_COL].values
sid_te  = test_df[STORM_COL].values
 
# Push tabular tensors to GPU (ANN & Log-Reg eval)
X_tr_gpu      = torch.tensor(X_tr,      device=PRIMARY)
y_tr_gpu      = torch.tensor(y_tr,      device=PRIMARY)
X_val_gpu     = torch.tensor(X_val,     device=PRIMARY)
X_te_gpu      = torch.tensor(X_te,      device=PRIMARY)
y_val_raw_gpu = torch.tensor(y_val_raw, device=PRIMARY)
y_te_raw_gpu  = torch.tensor(y_te_raw,  device=PRIMARY)
 
Y_MEAN = torch.tensor(scaler_y.mean_,  device=PRIMARY, dtype=torch.float32)
Y_STD  = torch.tensor(scaler_y.scale_, device=PRIMARY, dtype=torch.float32)
 
def descale(p: torch.Tensor) -> torch.Tensor:
    """Invert standard-scaling on predicted depth."""
    return p * Y_STD + Y_MEAN
 
print(f"✅ Tensors on {PRIMARY}. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

✅ Tensors on cuda:0. VRAM used: 0.16 GB


In [21]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 4 │ LSTM Storm-Safe Window Builder
# ─────────────────────────────────────────────────────────────────────────────
# Windows must NOT span storm boundaries. A window bridging the end of one
# storm and the start of the next would corrupt the temporal context fed to
# the LSTM with irrelevant inter-event data.
 
def build_storm_windows(X: np.ndarray, y: np.ndarray,
                        storm_ids: np.ndarray,
                        window: int):
    """
    Returns (X_windows, y_targets) where each row in X_windows is a
    (window, n_features) sequence drawn from a single storm, and the
    corresponding y_target is the depth at timestep t+1 after the window.
 
    Storms shorter than (window + 1) are skipped — they cannot contribute
    a full window without contaminating the boundary.
    """
    Xw, yw = [], []
    for sid in np.unique(storm_ids):
        mask = storm_ids == sid
        Xs, ys = X[mask], y[mask]
        n = len(Xs)
        if n <= window:
            continue
        for i in range(n - window):
            Xw.append(Xs[i : i + window])
            yw.append(ys[i + window])
    if len(Xw) == 0:
        return np.empty((0, window, X.shape[1]), dtype='float32'), np.empty((0, 1), dtype='float32')
    return (np.array(Xw, dtype='float32'),
            np.array(yw, dtype='float32').reshape(-1, 1))

In [22]:
def get_windows(split: str, window: int):
    """Return CPU tensors to save VRAM. Windows move to GPU batch-by-batch."""
    key = (split, window)
    if key not in _WINDOW_CACHE:
        if split == 'train':
            Xw, yw = build_storm_windows(X_tr, y_tr, sid_tr, window)
        elif split == 'val':
            Xw, yw = build_storm_windows(X_val, y_val, sid_val, window)
        else:
            y_te_sc = scaler_y.transform(test_df[[TARGET]]).astype('float32')
            Xw, yw  = build_storm_windows(X_te, y_te_sc, sid_te, window)
        
        # Store on CPU (default)
        _WINDOW_CACHE[key] = (
            torch.tensor(Xw), 
            torch.tensor(yw),
        )
    return _WINDOW_CACHE[key]
 

In [23]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 5 │ Model Architectures
# ─────────────────────────────────────────────────────────────────────────────
 
class ResidualBlock(nn.Module):
    """Pre-norm residual block: LayerNorm → Linear → ReLU → Dropout + skip."""
    def __init__(self, size: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(size)
        self.fc   = nn.Linear(size, size)
        self.drop = nn.Dropout(dropout)
 
    def forward(self, x):
        return x + self.drop(F.relu(self.fc(self.norm(x))))
 
 
class SotaANN(nn.Module):
    """
    Deep residual MLP for tabular flood prediction.
    Input: (B, n_features) → Output: (B, 1) scaled depth.
    Skip connections stabilise training depth up to 6+ layers.
    """
    def __init__(self, input_size: int, hidden_size: int,
                 n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.proj   = nn.Linear(input_size, hidden_size)
        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_size, dropout) for _ in range(n_layers)]
        )
        self.head = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        return self.head(self.blocks(F.relu(self.proj(x))))
 
 
class SotaAttentionLSTM(nn.Module):
    """
    Bidirectional LSTM + soft attention for sequence-to-scalar flood prediction.
    Input: (B, T, n_features) → Output: (B, 1) scaled depth.
    Attention weights allow the model to focus on the most hydrologically
    informative timesteps within each window (e.g., peak intensity).
    Bidirectionality helps leverage the full intra-storm context.
    """
    def __init__(self, input_size: int, hidden_size: int,
                 n_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        lstm_drop = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers=n_layers, batch_first=True,
                            bidirectional=True, dropout=lstm_drop)
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.norm = nn.LayerNorm(hidden_size * 2)
        self.head = nn.Linear(hidden_size * 2, 1)
 
    def forward(self, x):
        out, _   = self.lstm(x)                          # (B, T, 2H)
        weights  = F.softmax(self.attn(out), dim=1)      # (B, T, 1)
        context  = torch.sum(out * weights, dim=1)        # (B, 2H)
        return self.head(self.norm(context))
 
 
def wrap_model(model: nn.Module) -> nn.Module:
    """Apply DataParallel across all available GPUs and move to PRIMARY."""
    if N_GPUS > 1:
        model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
    return model.to(PRIMARY)

In [24]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 6 │ Hydrological Performance Metrics
# ─────────────────────────────────────────────────────────────────────────────
# Using the standard hydrological evaluation suite rather than RMSE alone.
 
def nse(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """
    Nash–Sutcliffe Efficiency. NSE = 1 is perfect; NSE < 0 means the mean
    observed is a better predictor than the model (unacceptable).
    """
    num = torch.sum((y_true - y_pred) ** 2)
    den = torch.sum((y_true - y_true.mean()) ** 2) + 1e-9
    return (1 - num / den).item()
 
 
def kge(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Kling–Gupta Efficiency. Decomposes error into correlation (r),
    variability bias (α), and mean bias (β). KGE = 1 is perfect.
    Preferred over NSE for flood peak assessment.
    """
    r     = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = y_pred.std()  / (y_true.std()  + 1e-9)
    beta  = y_pred.mean() / (y_true.mean() + 1e-9)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
 
 
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
 
 
def pbias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Percent Bias. 0% = perfect. Positive = model underestimates volume
    (dangerous for flood risk); negative = overestimates.
    """
    return float(100 * np.sum(y_true - y_pred) / (np.sum(y_true) + 1e-9))
 
 
def eval_metrics(name: str, y_true_np: np.ndarray,
                 y_pred_np: np.ndarray) -> dict:
    yt = torch.tensor(y_true_np, device=PRIMARY)
    yp = torch.tensor(y_pred_np, device=PRIMARY)
    return {
        'Model': name,
        'NSE':   round(nse(yt, yp), 4),
        'KGE':   round(kge(y_true_np, y_pred_np), 4),
        'RMSE':  round(rmse(y_true_np, y_pred_np), 4),
        'PBIAS': round(pbias(y_true_np, y_pred_np), 2),
    }

In [25]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 7 │ Optuna Objectives  (Val NSE is the optimisation target)
# ─────────────────────────────────────────────────────────────────────────────
# All objectives report VAL performance — never test. Test set is untouched
# until Block 10.
 
# ── 7a. Log-Space Ridge Regression (CPU) ────────────────────────────────────
# Log-transforming the target is hydrologically motivated: flood depths are
# log-normally distributed; the transform stabilises variance and prevents
# negative depth predictions.
 
def objective_log_reg(trial):
    alpha     = trial.suggest_float("alpha",     1e-3, 20.0, log=True)
    log_shift = trial.suggest_float("log_shift", 1e-4,  1.0, log=True)
 
    y_tr_log = np.log(train_df[TARGET] + log_shift)
    model    = Ridge(alpha=alpha).fit(train_df[FEATURES], y_tr_log)
 
    preds_val = np.exp(model.predict(val_df[FEATURES])) - log_shift
    y_val_np  = val_df[TARGET].values
    denom     = np.sum((y_val_np - y_val_np.mean()) ** 2) + 1e-9
    return float(1 - np.sum((y_val_np - preds_val) ** 2) / denom)
 

In [26]:
# ── 7b. Residual ANN (GPU, DataParallel) ────────────────────────────────────
# HuberLoss is more robust than MSE to the large depth spikes during extreme
# events, which would otherwise dominate the gradient signal.

def objective_ann(trial):
    h_size   = trial.suggest_int("hidden_size",  128,  512, step=64)
    n_layers = trial.suggest_int("n_layers",       2,    4)
    lr       = trial.suggest_float("lr",         1e-4, 5e-3, log=True)
    dropout  = trial.suggest_float("dropout",     0.0,  0.3)
    
    # CRITICAL FIX: Lowered from 262,144 to 32,768 for 11GB VRAM safety
    batch_sz = 32768   
 
    model   = wrap_model(SotaANN(len(FEATURES), h_size, n_layers, dropout))
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)
    loss_fn = nn.HuberLoss()
 
    best_val, patience, wait = float('inf'), 5, 0
 
    for _ in range(25):
        model.train()
        perm = torch.randperm(len(X_tr_gpu), device=PRIMARY)
        for i in range(0, len(X_tr_gpu), batch_sz):
            idx = perm[i : i + batch_sz]
            opt.zero_grad(set_to_none=True)
            with autocast(device_type='cuda'):
                loss = loss_fn(model(X_tr_gpu[idx]), y_tr_gpu[idx])
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()
        sched.step()
 
        model.eval()
        with torch.no_grad(), autocast(device_type='cuda'):
            val_nse = nse(y_val_raw_gpu, descale(model(X_val_gpu)).flatten())
 
        val_loss = -val_nse
        if val_loss < best_val - 1e-4:
            best_val, wait = val_loss, 0
        else:
            wait += 1
            if wait >= patience:
                break
                
    # Free memory before the next trial
    del model, opt
    torch.cuda.empty_cache()
 
    return -best_val

In [27]:
def objective_lstm(trial):
    window   = trial.suggest_int("window_size",  30,  90, step=30)
    h_size   = trial.suggest_int("hidden_size",  32, 128, step=32)
    n_layers = trial.suggest_int("n_layers",      1,   3)
    lr       = trial.suggest_float("lr",        1e-4, 5e-3, log=True)
    dropout  = trial.suggest_float("dropout",    0.0,  0.2)
    batch_sz = 2048 # Safe size for 11GB cards
 
    Xtw_cpu, ytw_cpu = get_windows('train', window)
    Xvw_cpu, yvw_cpu = get_windows('val',   window)
    
    if len(Xtw_cpu) == 0: return float('-inf')
 
    model = wrap_model(SotaAttentionLSTM(len(FEATURES), h_size, n_layers, dropout))
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.HuberLoss()
 
    for _ in range(15): # Reduced epochs for faster tuning
        model.train()
        perm = torch.randperm(len(Xtw_cpu))
        for i in range(0, len(Xtw_cpu), batch_sz):
            idx = perm[i : i + batch_sz]
            
            # MOVE MINI-BATCH TO GPU HERE
            bx = Xtw_cpu[idx].to(PRIMARY, non_blocking=True)
            by = ytw_cpu[idx].to(PRIMARY, non_blocking=True)
            
            opt.zero_grad(set_to_none=True)
            with autocast(device_type='cuda'):
                loss = loss_fn(model(bx), by)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()
 
        model.eval()
        # Validation evaluation also needs batching to avoid OOM
        with torch.no_grad():
            all_p = []
            for i in range(0, len(Xvw_cpu), batch_sz):
                bxv = Xvw_cpu[i : i + batch_sz].to(PRIMARY)
                all_p.append(model(bxv))
            
            preds_s = torch.cat(all_p)
            val_nse = nse(descale(yvw_cpu.to(PRIMARY)).flatten(), 
                          descale(preds_s).flatten())
            
    # Cleanup within trial
    del model, opt
    torch.cuda.empty_cache()
    return val_nse

In [ ]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 8 │ Hyperparameter Search
# ─────────────────────────────────────────────────────────────────────────────
N_TRIALS_LR   = 40
N_TRIALS_ANN  = 35
N_TRIALS_LSTM = 15
 
print("🔎 [1/3] Log-Ridge baseline …")
study_lr = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_lr.optimize(objective_log_reg, n_trials=N_TRIALS_LR)
 
print("🔎 [2/3] Residual ANN …")
study_ann = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_ann.optimize(objective_ann, n_trials=N_TRIALS_ANN)
# Clear the ANN from GPU before starting LSTM
if 'study_ann' in locals():
    del X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_raw_gpu
    torch.cuda.empty_cache()
    gc.collect()
    print("🧹 GPU Memory cleared for LSTM study.")
print("🔎 [3/3] Attention-LSTM …")
study_lstm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_lstm.optimize(objective_lstm, n_trials=N_TRIALS_LSTM)
 
print(f"\n{'─'*45}")
print(f"{'Model':<20} {'Val NSE':>10}")
print(f"{'─'*45}")
print(f"{'Log-Ridge':20} {study_lr.best_value:>10.4f}")
print(f"{'Res-ANN':20} {study_ann.best_value:>10.4f}")
print(f"{'Attn-LSTM':20} {study_lstm.best_value:>10.4f}")
print(f"{'─'*45}")

🔎 [1/3] Log-Ridge baseline …
🔎 [2/3] Residual ANN …


In [ ]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 9 │ Final Retraining on Train + Val with Best Hyperparameters
# ─────────────────────────────────────────────────────────────────────────────
# After hyperparameter selection we pool train + val for final fitting.
# The test set is still untouched. Scalers are re-fitted on the combined set
# so normalisation reflects the full non-test distribution.
 
print("\n🏋️  Retraining final models on Train + Val …")
 
train_val_df = pd.concat([train_df, val_df])
sid_tv       = train_val_df[STORM_COL].values
 
scaler_X2 = StandardScaler()
scaler_y2 = StandardScaler()
 
X_tv       = scaler_X2.fit_transform(train_val_df[FEATURES]).astype('float32')
y_tv       = scaler_y2.fit_transform(train_val_df[[TARGET]]).astype('float32')
X_te_final = scaler_X2.transform(test_df[FEATURES]).astype('float32')
 
X_tv_gpu       = torch.tensor(X_tv,       device=PRIMARY)
y_tv_gpu       = torch.tensor(y_tv,       device=PRIMARY)
X_te_final_gpu = torch.tensor(X_te_final, device=PRIMARY)
 
Y_MEAN2 = torch.tensor(scaler_y2.mean_,  device=PRIMARY, dtype=torch.float32)
Y_STD2  = torch.tensor(scaler_y2.scale_, device=PRIMARY, dtype=torch.float32)
 
def descale2(p: torch.Tensor) -> torch.Tensor:
    return p * Y_STD2 + Y_MEAN2
 
loss_fn = nn.HuberLoss()

In [ ]:
# ── 9a. Log-Ridge ─────────────────────────────────────────────────────────
bp       = study_lr.best_params
lr_final = Ridge(alpha=bp['alpha']).fit(
    train_val_df[FEATURES],
    np.log(train_val_df[TARGET] + bp['log_shift'])
)
lr_preds = np.exp(lr_final.predict(test_df[FEATURES])) - bp['log_shift']
print("   ✅ Log-Ridge done.")

In [ ]:
# ── 9b. Residual ANN ──────────────────────────────────────────────────────
ba         = study_ann.best_params
# CRITICAL FIX: Match the safe batch size from the objective function
BATCH_ANN  = 32768 
EPOCHS_ANN = 40
 
ann_final = wrap_model(SotaANN(len(FEATURES), ba['hidden_size'],
                               ba['n_layers'], ba['dropout']))
opt_ann   = optim.AdamW(ann_final.parameters(), lr=ba['lr'], weight_decay=1e-4)
sched_ann = optim.lr_scheduler.CosineAnnealingLR(opt_ann, T_max=EPOCHS_ANN)
 
ann_final.train()
for epoch in range(EPOCHS_ANN):
    perm = torch.randperm(len(X_tv_gpu), device=PRIMARY)
    for i in range(0, len(X_tv_gpu), BATCH_ANN):
        idx = perm[i : i + BATCH_ANN]
        opt_ann.zero_grad(set_to_none=True)
        with autocast(device_type='cuda'):
            loss = loss_fn(ann_final(X_tv_gpu[idx]), y_tv_gpu[idx])
        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt_ann)
        scaler_amp.update()
    sched_ann.step()
 
ann_final.eval()
with torch.no_grad():
    ann_preds = descale2(ann_final(X_te_final_gpu)).cpu().numpy().flatten()
 
print("   ✅ Res-ANN done.")

In [ ]:
# ── 9c. Attention-LSTM ────────────────────────────────────────────────────
bl           = study_lstm.best_params
WINDOW_FINAL = bl['window_size']
BATCH_LSTM   = 2048 # CRITICAL FIX: Lowered for 11GB VRAM safety
EPOCHS_LSTM  = 30

# Build storm-safe windows for train+val and test using updated scaler
y_tv_for_win     = scaler_y2.transform(train_val_df[[TARGET]]).astype('float32')
y_te_sc2         = scaler_y2.transform(test_df[[TARGET]]).astype('float32')
 
Xtv_w, ytv_w = build_storm_windows(X_tv, y_tv_for_win, sid_tv, WINDOW_FINAL)
Xte_w, yte_w = build_storm_windows(X_te_final, y_te_sc2, sid_te, WINDOW_FINAL)
 
Xtv_w_gpu = torch.tensor(Xtv_w, device=PRIMARY)
ytv_w_gpu = torch.tensor(ytv_w, device=PRIMARY)
Xte_w_gpu = torch.tensor(Xte_w, device=PRIMARY)
 
lstm_final = wrap_model(SotaAttentionLSTM(len(FEATURES), bl['hidden_size'],
                                          bl['n_layers'],  bl['dropout']))
opt_lstm   = optim.AdamW(lstm_final.parameters(), lr=bl['lr'], weight_decay=1e-4)
sched_lstm = optim.lr_scheduler.CosineAnnealingLR(opt_lstm, T_max=EPOCHS_LSTM)
 
lstm_final.train()
for epoch in range(EPOCHS_LSTM):
    perm = torch.randperm(len(Xtv_w_gpu), device=PRIMARY)
    for i in range(0, len(Xtv_w_gpu), BATCH_LSTM):
        idx = perm[i : i + BATCH_LSTM]
        opt_lstm.zero_grad(set_to_none=True)
        with autocast(device_type='cuda'):
            loss = loss_fn(lstm_final(Xtv_w_gpu[idx]), ytv_w_gpu[idx])
        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt_lstm)
        scaler_amp.update()
    sched_lstm.step()
 
lstm_final.eval()
with torch.no_grad():
    lstm_preds_s = torch.cat(
        [lstm_final(Xte_w_gpu[i : i + BATCH_LSTM])
         for i in range(0, len(Xte_w_gpu), BATCH_LSTM)]
    )
    lstm_preds = descale2(lstm_preds_s).cpu().numpy().flatten()
 
# Ground-truth depths aligned to LSTM windows (leading `window` rows per storm removed)
lstm_obs = descale2(torch.tensor(yte_w, device=PRIMARY)).cpu().numpy().flatten()
 
print("   ✅ Attn-LSTM done.")
print(f"\n✅ All models retrained on Train+Val ({len(train_val_df):,} rows, "
      f"{len(storm_ids[:n_tr + n_va])} storms).")

In [ ]:
# Build storm-safe windows for train+val and test using updated scaler
y_tv_for_win     = scaler_y2.transform(train_val_df[[TARGET]]).astype('float32')
y_te_sc2         = scaler_y2.transform(test_df[[TARGET]]).astype('float32')
 
Xtv_w, ytv_w = build_storm_windows(X_tv, y_tv_for_win, sid_tv, WINDOW_FINAL)
Xte_w, yte_w = build_storm_windows(X_te_final, y_te_sc2, sid_te, WINDOW_FINAL)
 
Xtv_w_gpu = torch.tensor(Xtv_w, device=PRIMARY)
ytv_w_gpu = torch.tensor(ytv_w, device=PRIMARY)
Xte_w_gpu = torch.tensor(Xte_w, device=PRIMARY)
 
lstm_final = wrap_model(SotaAttentionLSTM(len(FEATURES), bl['hidden_size'],
                                          bl['n_layers'],  bl['dropout']))
opt_lstm   = optim.AdamW(lstm_final.parameters(), lr=bl['lr'], weight_decay=1e-4)
sched_lstm = optim.lr_scheduler.CosineAnnealingLR(opt_lstm, T_max=EPOCHS_LSTM)
 
lstm_final.train()
for epoch in range(EPOCHS_LSTM):
    perm = torch.randperm(len(Xtv_w_gpu), device=PRIMARY)
    for i in range(0, len(Xtv_w_gpu), BATCH_LSTM):
        idx = perm[i : i + BATCH_LSTM]
        opt_lstm.zero_grad(set_to_none=True)
        with autocast(device_type='cuda'):
            loss = loss_fn(lstm_final(Xtv_w_gpu[idx]), ytv_w_gpu[idx])
        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt_lstm)
        scaler_amp.update()
    sched_lstm.step()
 
lstm_final.eval()
with torch.no_grad():
    lstm_preds_s = torch.cat(
        [lstm_final(Xte_w_gpu[i : i + BATCH_LSTM])
         for i in range(0, len(Xte_w_gpu), BATCH_LSTM)]
    )
    lstm_preds = descale2(lstm_preds_s).cpu().numpy().flatten()
 
# Ground-truth depths aligned to LSTM windows (leading `window` rows per storm removed)
lstm_obs = descale2(torch.tensor(yte_w, device=PRIMARY)).cpu().numpy().flatten()
 
print("   ✅ Attn-LSTM done.")
print(f"\n✅ All models retrained on Train+Val ({len(train_val_df):,} rows, "
      f"{len(storm_ids[:n_tr + n_va])} storms).")

In [ ]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 10 │ Test-Set Metrics  (first and only time test data is evaluated)
# ─────────────────────────────────────────────────────────────────────────────
metrics = [
    eval_metrics("Log-Ridge", y_te_raw,    lr_preds),
    eval_metrics("Res-ANN",   y_te_raw,    ann_preds),
    eval_metrics("Attn-LSTM", lstm_obs,    lstm_preds),
]
metrics_df = pd.DataFrame(metrics).set_index('Model')
 
print("\n📊 ── Final Test-Set Metrics ──────────────────────────")
print(f"{'':20} {'NSE':>8} {'KGE':>8} {'RMSE(in)':>10} {'PBIAS%':>8}")
print(f"{'─'*56}")
for name, row in metrics_df.iterrows():
    print(f"{name:20} {row['NSE']:>8.4f} {row['KGE']:>8.4f} "
          f"{row['RMSE']:>10.4f} {row['PBIAS']:>8.2f}")
print(f"{'─'*56}")
print("  NSE/KGE: 1=perfect | PBIAS: 0%=no bias, +=under, -=over")

In [ ]:
# %%─────────────────────────────────────────────────────────────────────────
# BLOCK 11 │ Visualisation
# ─────────────────────────────────────────────────────────────────────────────
COLORS = {
    'obs':  '#1a1a2e',
    'lr':   '#e67e22',
    'ann':  '#8e44ad',
    'lstm': '#16a085',
    'grid': '#cccccc',
}
 
fig = plt.figure(figsize=(18, 16), dpi=150)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.32)
 
# ── Helper: pick a representative test storm with enough rows for plotting ──
def pick_display_storm(min_rows=60):
    for sid in test_storms:
        n = (test_df[STORM_COL] == sid).sum()
        if n >= min_rows:
            return sid
    return test_storms[0]   # fallback
 
focus_sid  = pick_display_storm()
storm_mask = test_df[STORM_COL].values == focus_sid
storm_obs  = test_df.loc[test_df[STORM_COL] == focus_sid, TARGET].values
storm_lr   = lr_preds[storm_mask]
storm_ann  = ann_preds[storm_mask]
t_axis     = np.arange(len(storm_obs))
 

In [ ]:
# ── Panel A: ANN & Log-Ridge storm time-series ───────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(t_axis, 0, storm_obs, color=COLORS['obs'], alpha=0.12)
ax1.plot(t_axis, storm_obs, label='Observed (FloodNet)',
         color=COLORS['obs'], lw=2.5, alpha=0.95, zorder=3)
ax1.plot(t_axis, storm_lr,
         label=f"Log-Ridge  NSE={metrics_df.loc['Log-Ridge','NSE']:.3f}  "
               f"KGE={metrics_df.loc['Log-Ridge','KGE']:.3f}",
         color=COLORS['lr'], ls='--', lw=1.8, zorder=2)
ax1.plot(t_axis, storm_ann,
         label=f"Res-ANN    NSE={metrics_df.loc['Res-ANN','NSE']:.3f}  "
               f"KGE={metrics_df.loc['Res-ANN','KGE']:.3f}",
         color=COLORS['ann'], lw=2.0, zorder=2)
ax1.set_title(f"Storm Event Comparison — Storm ID: {focus_sid}",
              fontsize=13, fontweight='bold')
ax1.set_ylabel("Water Depth (in)", fontsize=11)
ax1.set_xlabel("Timestep (min)", fontsize=11)
ax1.legend(fontsize=9, framealpha=0.9)
ax1.grid(True, color=COLORS['grid'], alpha=0.5)

In [ ]:
# ── Panel B: LSTM storm segment ──────────────────────────────────────────
DISP = min(800, len(lstm_preds))
ax2  = fig.add_subplot(gs[1, :])
t2   = np.arange(DISP)
ax2.fill_between(t2, 0, lstm_obs[:DISP], color=COLORS['obs'], alpha=0.12)
ax2.plot(t2, lstm_obs[:DISP],   label='Observed (FloodNet)',
         color=COLORS['obs'],  lw=2.5, alpha=0.95, zorder=3)
ax2.plot(t2, lstm_preds[:DISP],
         label=f"Attn-LSTM  NSE={metrics_df.loc['Attn-LSTM','NSE']:.3f}  "
               f"KGE={metrics_df.loc['Attn-LSTM','KGE']:.3f}",
         color=COLORS['lstm'], lw=2.0, zorder=2)
ax2.set_title("Attention-LSTM — Test Set Segment",
              fontsize=13, fontweight='bold')
ax2.set_ylabel("Water Depth (in)", fontsize=11)
ax2.set_xlabel("Timestep (min)", fontsize=11)
ax2.legend(fontsize=9, framealpha=0.9)
ax2.grid(True, color=COLORS['grid'], alpha=0.5)

In [ ]:
# ── Panel C: Scatter — Observed vs ANN Predicted ────────────────────────
ax3  = fig.add_subplot(gs[2, 0])
lim  = (min(y_te_raw.min(), ann_preds.min()) * 0.95,
        max(y_te_raw.max(), ann_preds.max()) * 1.05)
ax3.scatter(y_te_raw, ann_preds, alpha=0.12, s=3,
            color=COLORS['ann'], rasterized=True)
ax3.plot(lim, lim, 'k--', lw=1.2, label='1:1 line')
ax3.set_xlim(lim); ax3.set_ylim(lim)
ax3.set_title("Res-ANN: Observed vs Predicted", fontsize=12, fontweight='bold')
ax3.set_xlabel("Observed (in)", fontsize=10)
ax3.set_ylabel("Predicted (in)", fontsize=10)
ax3.legend(fontsize=9); ax3.grid(True, color=COLORS['grid'], alpha=0.5)

In [ ]:
# ── Panel D: Grouped metric bars (NSE & KGE per model) ──────────────────
ax4    = fig.add_subplot(gs[2, 1])
models = metrics_df.index.tolist()
x_pos  = np.arange(len(models))
bw     = 0.32
b1 = ax4.bar(x_pos - bw / 2, metrics_df['NSE'], bw,
             label='NSE', color='#2980b9', alpha=0.87)
b2 = ax4.bar(x_pos + bw / 2, metrics_df['KGE'], bw,
             label='KGE', color='#c0392b', alpha=0.87)
ax4.axhline(0,   color='black', lw=0.8, ls='--', alpha=0.5)
ax4.axhline(0.5, color='green', lw=0.8, ls=':',  alpha=0.6, label='NSE/KGE = 0.5 (satisfactory)')
ax4.axhline(0.65,color='green', lw=0.8, ls='--', alpha=0.4, label='NSE/KGE = 0.65 (good)')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(models, fontsize=10)
ax4.set_ylim(-0.15, 1.08)
ax4.set_ylabel("Score  (1 = perfect)", fontsize=10)
ax4.set_title("Model Performance: NSE & KGE", fontsize=12, fontweight='bold')
ax4.legend(fontsize=8, loc='lower right', framealpha=0.9)
ax4.grid(True, color=COLORS['grid'], alpha=0.5, axis='y')

In [ ]:
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width() / 2, h + 0.012,
             f"{h:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')
 
fig.suptitle("NYC FloodNet — Flood Depth Prediction Model Shootout\n"
             "(Storm-aware CV  ·  Train / Val / Test  ·  2-GPU DataParallel)",
             fontsize=14, fontweight='bold', y=1.01)
 
OUT_FIG = 'Images_or_plots/flood_model_shootout.png'
plt.savefig(OUT_FIG, bbox_inches='tight', dpi=150)
plt.show()
print(f"✅ Figure saved → {OUT_FIG}")